In [1]:
import numpy as np
import pandas as pd
from collections import Counter

## Prepare annotations for baseline

In [2]:
def iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area = lambda b: (b[2]-b[0])*(b[3]-b[1])
    return inter / (area(box1) + area(box2) - inter + 1e-8)

def build_iou_matrix(bboxes):
    n = len(bboxes)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            mat[i,j] = mat[j,i] = iou(bboxes[i], bboxes[j])
    return mat

def greedy_cluster(iou_mat, thresh=0.5):
    n = len(iou_mat)
    used = [False]*n
    clusters = []
    pairs = [(iou_mat[i,j], i, j) for i in range(n) for j in range(i+1,n) if iou_mat[i,j] >= thresh]
    pairs.sort(reverse=True, key=lambda x: x[0])
    
    for _, i, j in pairs:
        if used[i] or used[j]:
            continue
        cluster = [i, j]
        used[i] = used[j] = True
        changed = True
        while changed:
            changed = False
            for k in range(n):
                if not used[k] and any(iou_mat[k, m] >= thresh for m in cluster):
                    cluster.append(k); used[k] = True; changed = True
        clusters.append(cluster)

    for i in range(n):
        if not used[i]:
            clusters.append([i])
    return clusters

def aggregate_cluster(ann_list, indices):
    classes = [ann_list[i]['class'] for i in indices]
    bboxes = [ann_list[i]['bbox'] for i in indices]

    consensus_class = Counter(classes).most_common(1)[0][0]

    consensus_bbox = np.mean(bboxes, axis=0).tolist()
    return consensus_class, consensus_bbox

# --------------------- Основная функция (только управление) ---------------------

def create_baseline_consensus(df_raw, iou_thresh=0.5, save_path=None):
    """
    Создаёт baseline консенсус: Majority Vote для класса, средний бокс.
    Возвращает DataFrame с колонками: image_id, class_id, x_min, y_min, x_max, y_max
    """
    rows = []
    for image_id, group in df_raw.groupby('image_id'):
        # Преобразуем строки группы в список словарей
        ann_items = [{'rad_id': r['rad_id'], 'class': r['class_id'], 
                      'bbox': [r['x_min'], r['y_min'], r['x_max'], r['y_max']]} 
                     for _, r in group.iterrows()]
        if not ann_items:
            continue
        # Матрица IoU и кластеры
        iou_mat = build_iou_matrix([a['bbox'] for a in ann_items])
        clusters = greedy_cluster(iou_mat, thresh=iou_thresh)
        # Агрегация каждого кластера
        for cl in clusters:
            cls, bbox = aggregate_cluster(ann_items, cl)
            rows.append([image_id, cls, *bbox])
    df = pd.DataFrame(rows, columns=['image_id', 'class_id', 'x_min', 'y_min', 'x_max', 'y_max'])
    if save_path:
        df.to_csv(save_path, index=False)
        print(f"✅ Сохранено {len(df)} объектов в {save_path}")
    return df

In [4]:
df = pd.read_csv(r"data/VinDr-CXR/images/train.csv")

In [5]:
save_path = r"data/VinDr-CXR/images/baseline_labels.csv"
bl_df = create_baseline_consensus(df, iou_thresh=0.4, save_path=save_path)

✅ Сохранено 49489 объектов в data/VinDr-CXR/images/baseline_labels.csv
